In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS aira_test.aibi_workshop_schema;

In [0]:
%sql
CREATE OR REPLACE VIEW aira_test.aibi_workshop_schema.tpch_analytics_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "TPC-H analytics metric view for relationship analytics dashboard"

source: samples.tpch.lineitem

joins:
  - name: orders
    source: samples.tpch.orders
    "on": source.l_orderkey = orders.o_orderkey
    joins:
      - name: customer
        source: samples.tpch.customer
        "on": orders.o_custkey = customer.c_custkey
        joins:
          - name: customer_nation
            source: samples.tpch.nation
            "on": customer.c_nationkey = customer_nation.n_nationkey
            joins:
              - name: customer_region
                source: samples.tpch.region
                "on": customer_nation.n_regionkey = customer_region.r_regionkey
  - name: supplier
    source: samples.tpch.supplier
    "on": source.l_suppkey = supplier.s_suppkey
    joins:
      - name: supplier_nation
        source: samples.tpch.nation
        "on": supplier.s_nationkey = supplier_nation.n_nationkey
        joins:
          - name: supplier_region
            source: samples.tpch.region
            "on": supplier_nation.n_regionkey = supplier_region.r_regionkey

dimensions:
  - name: order_month
    expr: "DATE_TRUNC('month', orders.o_orderdate)"
    display_name: 'Order Month'
    comment: 'Month in which the order was placed'
    synonyms: ['month', 'order month']
    format:
      type: date
      date_format: locale_short_month

  - name: order_date
    expr: orders.o_orderdate
    display_name: 'Order Date'
    comment: 'Date the order was placed'

  - name: order_status
    expr: |-
      CASE
        WHEN orders.o_orderstatus = 'O' THEN 'Open'
        WHEN orders.o_orderstatus = 'P' THEN 'Processing'
        WHEN orders.o_orderstatus = 'F' THEN 'Fulfilled'
      END
    display_name: 'Order Status'
    comment: 'Status of the order'

  - name: order_priority
    expr: orders.o_orderpriority
    display_name: 'Order Priority'
    comment: 'Priority level of the order'

  - name: customer_name
    expr: orders.customer.c_name
    display_name: 'Customer Name'
    comment: 'Name of the customer'

  - name: market_segment
    expr: orders.customer.c_mktsegment
    display_name: 'Market Segment'
    comment: 'Market segment of the customer'

  - name: customer_nation
    expr: orders.customer.customer_nation.n_name
    display_name: 'Customer Nation'
    comment: 'Nation of the customer'
    synonyms: ['nation', 'country']

  - name: customer_region
    expr: orders.customer.customer_nation.customer_region.r_name
    display_name: 'Customer Region'
    comment: 'Region of the customer'
    synonyms: ['region']

  - name: supplier_name
    expr: supplier.s_name
    display_name: 'Supplier Name'
    comment: 'Name of the supplier'

  - name: supplier_nation
    expr: supplier.supplier_nation.n_name
    display_name: 'Supplier Nation'
    comment: 'Nation of the supplier'

  - name: supplier_region
    expr: supplier.supplier_nation.supplier_region.r_name
    display_name: 'Supplier Region'
    comment: 'Region of the supplier'

  - name: ship_mode
    expr: l_shipmode
    display_name: 'Ship Mode'
    comment: 'Shipping mode'

  - name: ship_instruction
    expr: l_shipinstruct
    display_name: 'Ship Instruction'
    comment: 'Shipping instruction'

measures:
  - name: revenue
    expr: SUM(source.l_extendedprice * (1 - source.l_discount))
    display_name: 'Revenue'
    comment: 'Total revenue calculated as extended price minus discount'
    synonyms: ['sales', 'total revenue']
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact

  - name: line_item_count
    expr: COUNT(*)
    display_name: 'Line Item Count'
    comment: 'Total number of line items'
    format:
      type: number
      decimal_places:
        type: exact
        places: 0

  - name: order_count
    expr: COUNT(DISTINCT orders.o_orderkey)
    display_name: 'Order Count'
    comment: 'Total number of unique orders'
    format:
      type: number
      decimal_places:
        type: exact
        places: 0

  - name: unique_customers
    expr: COUNT(DISTINCT orders.customer.c_custkey)
    display_name: 'Unique Customers'
    comment: 'Number of unique customers'
    format:
      type: number
      decimal_places:
        type: exact
        places: 0

  - name: avg_delivery_days
    expr: "AVG(DATEDIFF(source.l_receiptdate, source.l_shipdate))"
    display_name: 'Avg Delivery Days'
    comment: 'Average number of days from shipment to receipt'
    format:
      type: number
      decimal_places:
        type: exact
        places: 1
$$;

result


In [0]:
%sql
SELECT customer_region, MEASURE(revenue) AS revenue, MEASURE(order_count) AS order_count
FROM aira_test.aibi_workshop_schema.tpch_analytics_metrics
GROUP BY customer_region
ORDER BY revenue DESC

customer_region,revenue,order_count
EUROPE,218954324383.8806,1505888
MIDDLE EAST,218773014374.5862,1505370
ASIA,218445802361.8825,1504087
AMERICA,216983842403.1653,1493343
AFRICA,216678195723.7009,1491312


In [0]:
%sql
CREATE OR REPLACE VIEW `aira_test`.`aibi_workshop_schema`.`tpch_analytics_metrics_v1`
(
  `Order Month`,
  `Order Date`,
  `Order Status`,
  `Order Priority`,
  `Customer Name`,
  `Market Segment`,
  `Customer Nation`,
  `Customer Region`,
  `Supplier Name`,
  `Supplier Nation`,
  `Supplier Region`,
  `Ship Mode`,
  `Ship Instruction`,
  `Revenue`,
  `Line Item Count`,
  `Order Count`,
  `Unique Customers`,
  `Avg Delivery Days`
)
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1

source: >
  SELECT
    l.l_orderkey,
    l.l_extendedprice,
    l.l_discount,
    l.l_shipmode,
    l.l_shipinstruct,
    l.l_shipdate,
    l.l_receiptdate,
    o.o_orderkey,
    o.o_orderdate,
    o.o_orderstatus,
    o.o_orderpriority,
    c.c_custkey,
    c.c_name,
    c.c_mktsegment,
    cn.n_name AS customer_nation_name,
    cr.r_name AS customer_region_name,
    s.s_name,
    sn.n_name AS supplier_nation_name,
    sr.r_name AS supplier_region_name
  FROM samples.tpch.lineitem l
  LEFT JOIN samples.tpch.orders o
    ON l.l_orderkey = o.o_orderkey
  LEFT JOIN samples.tpch.customer c
    ON o.o_custkey = c.c_custkey
  LEFT JOIN samples.tpch.nation cn
    ON c.c_nationkey = cn.n_nationkey
  LEFT JOIN samples.tpch.region cr
    ON cn.n_regionkey = cr.r_regionkey
  LEFT JOIN samples.tpch.supplier s
    ON l.l_suppkey = s.s_suppkey
  LEFT JOIN samples.tpch.nation sn
    ON s.s_nationkey = sn.n_nationkey
  LEFT JOIN samples.tpch.region sr
    ON sn.n_regionkey = sr.r_regionkey

comment: TPC-H analytics metric view (v1 - SQL source joins)

dimensions:
  - name: order_month
    expr: "DATE_TRUNC('month', o_orderdate)"
    display_name: Order Month
    comment: Month in which the order was placed
    format:
      type: date
      date_format: locale_short_month
    synonyms: [month, order month]
  - name: order_date
    expr: o_orderdate
    display_name: Order Date
    comment: Date the order was placed
  - name: order_status
    expr: |-
      CASE
        WHEN o_orderstatus = 'O' THEN 'Open'
        WHEN o_orderstatus = 'P' THEN 'Processing'
        WHEN o_orderstatus = 'F' THEN 'Fulfilled'
      END
    display_name: Order Status
    comment: Status of the order
  - name: order_priority
    expr: o_orderpriority
    display_name: Order Priority
    comment: Priority level of the order
  - name: customer_name
    expr: c_name
    display_name: Customer Name
    comment: Name of the customer
  - name: market_segment
    expr: c_mktsegment
    display_name: Market Segment
    comment: Market segment of the customer
  - name: customer_nation
    expr: customer_nation_name
    display_name: Customer Nation
    comment: Nation of the customer
    synonyms: [nation, country]
  - name: customer_region
    expr: customer_region_name
    display_name: Customer Region
    comment: Region of the customer
    synonyms: [region]
  - name: supplier_name
    expr: s_name
    display_name: Supplier Name
    comment: Name of the supplier
  - name: supplier_nation
    expr: supplier_nation_name
    display_name: Supplier Nation
    comment: Nation of the supplier
  - name: supplier_region
    expr: supplier_region_name
    display_name: Supplier Region
    comment: Region of the supplier
  - name: ship_mode
    expr: l_shipmode
    display_name: Ship Mode
    comment: Shipping mode
  - name: ship_instruction
    expr: l_shipinstruct
    display_name: Ship Instruction
    comment: Shipping instruction

measures:
  - name: revenue
    expr: SUM(l_extendedprice * (1 - l_discount))
    display_name: Revenue
    comment: Total revenue calculated as extended price minus discount
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact
    synonyms: [sales, total revenue]
  - name: line_item_count
    expr: COUNT(*)
    display_name: Line Item Count
    comment: Total number of line items
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: order_count
    expr: COUNT(DISTINCT o_orderkey)
    display_name: Order Count
    comment: Total number of unique orders
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: unique_customers
    expr: COUNT(DISTINCT c_custkey)
    display_name: Unique Customers
    comment: Number of unique customers
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: avg_delivery_days
    expr: "AVG(DATEDIFF(l_receiptdate, l_shipdate))"
    display_name: Avg Delivery Days
    comment: Average number of days from shipment to receipt
    format:
      type: number
      decimal_places:
        type: exact
        places: 1
$$

result


In [0]:
%sql
CREATE OR REPLACE VIEW aira_test.aibi_workshop_schema.order_management_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "TPC-H order management metric view for relationship analytics with window measures"

source: aira_test.aibi_workshop_schema.lineitem

joins:
  - name: orders
    source: aira_test.aibi_workshop_schema.orders
    "on": source.l_orderkey = orders.o_orderkey
    joins:
      - name: customer
        source: aira_test.aibi_workshop_schema.customer
        "on": orders.o_custkey = customer.c_custkey
        joins:
          - name: customer_nation
            source: aira_test.aibi_workshop_schema.nation
            "on": customer.c_nationkey = customer_nation.n_nationkey
            joins:
              - name: customer_region
                source: aira_test.aibi_workshop_schema.region
                "on": customer_nation.n_regionkey = customer_region.r_regionkey
  - name: supplier
    source: aira_test.aibi_workshop_schema.supplier
    "on": source.l_suppkey = supplier.s_suppkey
    joins:
      - name: supplier_nation
        source: aira_test.aibi_workshop_schema.nation
        "on": supplier.s_nationkey = supplier_nation.n_nationkey
        joins:
          - name: supplier_region
            source: aira_test.aibi_workshop_schema.region
            "on": supplier_nation.n_regionkey = supplier_region.r_regionkey

dimensions:
  - name: order_month
    expr: "DATE_TRUNC('month', orders.o_orderdate)"
    display_name: 'Order Month'
    comment: 'Month in which the order was placed'
    synonyms: ['month', 'order month']
    format:
      type: date
      date_format: locale_short_month

  - name: order_date
    expr: orders.o_orderdate
    display_name: 'Order Date'
    comment: 'Date the order was placed'

  - name: order_status
    expr: |-
      CASE
        WHEN orders.o_orderstatus = 'O' THEN 'Open'
        WHEN orders.o_orderstatus = 'P' THEN 'Processing'
        WHEN orders.o_orderstatus = 'F' THEN 'Fulfilled'
      END
    display_name: 'Order Status'
    comment: 'Status of the order'

  - name: order_priority
    expr: orders.o_orderpriority
    display_name: 'Order Priority'
    comment: 'Priority level of the order'

  - name: customer_name
    expr: orders.customer.c_name
    display_name: 'Customer Name'
    comment: 'Name of the customer'

  - name: market_segment
    expr: orders.customer.c_mktsegment
    display_name: 'Market Segment'
    comment: 'Market segment of the customer'

  - name: customer_nation
    expr: orders.customer.customer_nation.n_name
    display_name: 'Customer Nation'
    comment: 'Nation of the customer'
    synonyms: ['nation', 'country']

  - name: customer_region
    expr: orders.customer.customer_nation.customer_region.r_name
    display_name: 'Customer Region'
    comment: 'Region of the customer'
    synonyms: ['region']

  - name: supplier_name
    expr: supplier.s_name
    display_name: 'Supplier Name'
    comment: 'Name of the supplier'

  - name: supplier_nation
    expr: supplier.supplier_nation.n_name
    display_name: 'Supplier Nation'
    comment: 'Nation of the supplier'

  - name: supplier_region
    expr: supplier.supplier_nation.supplier_region.r_name
    display_name: 'Supplier Region'
    comment: 'Region of the supplier'

  - name: ship_mode
    expr: l_shipmode
    display_name: 'Ship Mode'
    comment: 'Shipping mode'

  - name: ship_instruction
    expr: l_shipinstruct
    display_name: 'Ship Instruction'
    comment: 'Shipping instruction'

  - name: return_flag
    expr: l_returnflag
    display_name: 'Return Flag'
    comment: 'Return flag indicator'

measures:
  - name: revenue
    expr: SUM(source.l_extendedprice * (1 - source.l_discount))
    display_name: 'Revenue'
    comment: 'Total revenue calculated as extended price minus discount'
    synonyms: ['sales', 'total revenue']
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact

  - name: line_item_count
    expr: COUNT(*)
    display_name: 'Line Item Count'
    comment: 'Total number of line items'
    format:
      type: number
      decimal_places:
        type: exact
        places: 0

  - name: order_count
    expr: COUNT(DISTINCT orders.o_orderkey)
    display_name: 'Order Count'
    comment: 'Total number of unique orders'
    format:
      type: number
      decimal_places:
        type: exact
        places: 0

  - name: unique_customers
    expr: COUNT(DISTINCT orders.customer.c_custkey)
    display_name: 'Unique Customers'
    comment: 'Number of unique customers'
    format:
      type: number
      decimal_places:
        type: exact
        places: 0

  - name: total_quantity
    expr: SUM(source.l_quantity)
    display_name: 'Total Quantity'
    comment: 'Total quantity of items ordered'
    format:
      type: number
      decimal_places:
        type: exact
        places: 0

  - name: total_extended_price
    expr: SUM(source.l_extendedprice)
    display_name: 'Total Extended Price'
    comment: 'Sum of extended prices before discount'
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact

  - name: fulfillment_rate
    expr: >-
      COUNT(DISTINCT orders.o_orderkey) FILTER (WHERE orders.o_orderstatus = 'F')
      / COUNT(DISTINCT orders.o_orderkey)
    display_name: 'Fulfillment Rate'
    comment: 'Ratio of fulfilled orders to total orders'
    format:
      type: percentage
      decimal_places:
        type: exact
        places: 1

  - name: avg_delivery_days
    expr: "AVG(DATEDIFF(source.l_receiptdate, source.l_shipdate))"
    display_name: 'Avg Delivery Days'
    comment: 'Average number of days from shipment to receipt'
    format:
      type: number
      decimal_places:
        type: exact
        places: 1
$$;

result


In [0]:
%sql
CREATE OR REPLACE VIEW `aira_test`.`aibi_workshop_schema`.`order_management_window_measures_metrics`
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1

source: aira_test.aibi_workshop_schema.order_management_metrics

dimensions:
  - name: order_month
    expr: order_month
    comment: Month in which the order was placed
    display_name: Order Month
    format:
      type: date
      date_format: locale_short_month
  - name: order_date
    expr: order_date
    comment: Date in which the order was placed
    display_name: Order Date

measures:
  - name: trailing_7d_revenue
    expr: MEASURE(revenue)
    window:
      - order: order_date
        semiadditive: last
        range: trailing 7 day
    comment: Sum of revenue over the trailing 7 days window
    display_name: 7 Day Trailing Revenue
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact
  - name: trailing_30d_unique_customers
    expr: MEASURE(unique_customers)
    window:
      - order: order_month
        semiadditive: last
        range: trailing 30 day
    comment: Sum of unique customers over the trailing 30 days window
    display_name: 30 Day Trailing Unique Customers
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
$$

result
